# OKR Scoring Multi-Agent System

This notebook implements a LangGraph-based multi-agent system for scoring OKRs (Objectives and Key Results) and providing status updates with blocker identification.

## System Architecture

The system consists of 4 main agents:
1. **ACHIEVEMENT_SCORING_AGENT**: Updates status and scores each KR based on updates
2. **BLOCKER_IDENTIFICATION_AGENT**: Identifies blockers and challenges for each KR
3. **STATUS_REPORTER**: Creates output status and OKR score files
4. **SUPERVISOR**: Coordinates workflow between agents

## Input Data
- `<name>.okr.txt`: Objectives and KRs with percentage weights
- `<name>.status.txt`: Current status per KR
- `<name>.update.txt`: Recent updates with wins, challenges, next steps

## Output
- `<name>.status.txt`: Updated status file
- `<name>.okrscore.txt`: Objective scoring based on KR completion


## Setup and Dependencies

First, we'll set up our environment variables and import all necessary libraries.


## Installation

Install the required Python packages:


In [1]:
# Cell Tag: #install-packages
%pip install langchain langchain-openai langchain-community langgraph langsmith



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell Tag: #imports
import os
import getpass
from uuid import uuid4
from typing import TypedDict, Annotated, List
from pathlib import Path
import operator
import functools
import re
from datetime import datetime

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages


## Environment Configuration

Set up LangSmith tracing for monitoring and debugging our multi-agent system.

**LangSmith Setup Instructions:**
1. Go to [LangSmith](https://smith.langchain.com/) and create an account
2. Create a new project or use an existing one
3. Navigate to Settings → API Keys
4. Create a new API key and copy it
5. Enter the API key when prompted below

LangSmith will automatically trace all agent interactions, tool usage, and workflow execution, providing valuable insights into your multi-agent system's performance.


In [3]:
# Cell Tag: #langsmith-setup
# Set up LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"OKR-Scoring-System-{uuid4().hex[0:8]}"
#os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")


In [4]:
# Cell Tag: #openai-setup
# Set up OpenAI API key
#os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")


In [5]:
# Cell Tag: #env-setup
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

print("Environment variables loaded from .env file")
print(f"OPENAI_API_KEY configured: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")
print(f"LANGCHAIN_API_KEY configured: {'Yes' if os.getenv('LANGCHAIN_API_KEY') else 'No'}")


Environment variables loaded from .env file
OPENAI_API_KEY configured: Yes
LANGCHAIN_API_KEY configured: Yes


## Working Directory Setup

Set up directories for input data and output files.


In [6]:
# Cell Tag: #api-keys-setup
# Set up API keys (with .env file support)
# LangSmith setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"OKR-Scoring-System-{uuid4().hex[0:8]}"

# Check if API keys are loaded from .env file
langsmith_key = os.getenv("LANGCHAIN_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not langsmith_key:
    print("⚠️  LANGCHAIN_API_KEY not found in environment variables.")
    print("   Please add it to your .env file or set it manually:")
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")
else:
    print("✅ LangSmith API key loaded from .env file")

if not openai_key:
    print("⚠️  OPENAI_API_KEY not found in environment variables.")
    print("   Please add it to your .env file or set it manually:")
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
else:
    print("✅ OpenAI API key loaded from .env file")


✅ LangSmith API key loaded from .env file
✅ OpenAI API key loaded from .env file


In [7]:
# Cell Tag: #directory-config
# Working directories
DATA_DIRECTORY = Path("./data")
OUTPUT_DIRECTORY = Path("./output")
OUTPUT_DIRECTORY.mkdir(exist_ok=True)

# Create timestamped subdirectory for this session
from datetime import datetime
timestamp = datetime.now().strftime("%y%m%d-%H%M%S")
SESSION_OUTPUT_DIR = OUTPUT_DIRECTORY / timestamp
SESSION_OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data directory: {DATA_DIRECTORY.absolute()}")
print(f"Output directory: {OUTPUT_DIRECTORY.absolute()}")
print(f"Session output directory: {SESSION_OUTPUT_DIR.absolute()}")

# List available data files
data_files = list(DATA_DIRECTORY.glob("*.txt"))
print(f"\nAvailable data files: {[f.name for f in data_files]}")


Data directory: /Users/foohm/github/charlie/data
Output directory: /Users/foohm/github/charlie/output
Session output directory: /Users/foohm/github/charlie/output/251006-103613

Available data files: ['nancy.status.txt', 'walter.update.txt', 'nancy.okr.txt', 'joe.update.txt', 'walter.status.txt', 'joe.okr.txt', 'nancy.update.txt', 'walter.okr.txt', 'joe.status.txt']


## State Definition

Define the state that will be passed between agents. This includes the conversation history, OKR data, status updates, and workflow control.


In [ ]:
# Cell Tag: #agent-prompts
# Agent Prompts - Single Source of Truth
ACHIEVEMENT_SCORING_PROMPT = """You are an expert in OKR (Objectives and Key Results) achievement scoring. Your role is to analyze the OKR data, current status, and updates to:
1. Update the status of each Key Result based on the provided updates
2. Calculate completion percentages for each KR
3. Provide detailed scoring analysis

When analyzing updates, look for:
- Progress indicators and metrics mentioned
- Wins and achievements
- Completion percentages or progress descriptions
- Time-based progress indicators

Format your output as updated status for each KR with completion percentages.
Be precise and analytical in your scoring.
Make sure to only identify achievements based on the updates. Do not make up new achievements.
Explain how the status updates were arrived at based on the updates.

Work autonomously according to your specialty, using the tools available to you. Do not ask for clarification. Your other team members will collaborate with you with their own specialties. You are chosen for a reason!"""

BLOCKER_IDENTIFICATION_PROMPT = """You are an expert in identifying and analyzing blockers and challenges in OKR execution. Your role is to:
1. Analyze the OKR data, current status, and updates
2. Identify specific blockers and challenges for each Key Result
3. Provide reframed perspectives on these challenges
4. Suggest actionable solutions or mitigation strategies

Look for:
- Explicit challenges mentioned in updates
- Resource constraints (time, people, budget)
- Technical or process obstacles
- Dependencies and coordination issues
- External factors and risks

Provide clear, actionable blocker identification for each KR with reframed perspectives.
Make sure to only identify new blockers based on the updates. Do not make up new blockers or challenges.
Do not infer blockers from achievements.
Explain how the blocker was arrived at based on the updates.

Work autonomously according to your specialty, using the tools available to you. Do not ask for clarification. Your other team members will collaborate with you with their own specialties. You are chosen for a reason!"""

STATUS_REPORTER_PROMPT = """You are an expert in creating comprehensive OKR status reports. Your role is to:
1. Synthesize the updated status from the Achievement Scoring Agent
2. Incorporate blocker analysis from the Blocker Identification Agent
3. Calculate the overall objective score based on KR completion
4. Create two output files with specific naming:
   - Status file: '{person_name}.status.txt' (e.g., 'joe.status.txt')
   - Score file: '{person_name}.okrscore.txt' (e.g., 'joe.okrscore.txt')

Format the status file to include:
- Updated KR status with completion percentages
- Identified blockers and reframed perspectives
- Next steps and recommendations

Format the OKR score file to include:
- Overall objective score
- Individual KR scores and weights
- Scoring methodology and rationale

CRITICAL: For the "Individual KR Scores and Weights" section in the OKR score file, use EXACTLY this format:
### Individual KR Scores and Weights
KR1: [completion]% (Weight: [weight]%)
KR2: [completion]% (Weight: [weight]%)
KR3: [completion]% (Weight: [weight]%)

Example:
### Individual KR Scores and Weights
KR1: 76% (Weight: 20%)
KR2: 80% (Weight: 40%)
KR3: 87% (Weight: 40%)

IMPORTANT: Use the exact file naming convention above. Files will be automatically saved to the timestamped session directory.
Make sure to calculate the objective score based on the KR scores and weights.
For example, objective score = (KR1 score * KR1 weight + KR2 score * KR2 weight + ... + KRn score * KRn weight) / 100

Work autonomously according to your specialty, using the tools available to you. Do not ask for clarification. Your other team members will collaborate with you with their own specialties. You are chosen for a reason!"""

SUPERVISOR_PROMPT = """You are a supervisor coordinating a team to analyze and score OKRs. The team consists of: ACHIEVEMENT_SCORING_AGENT (updates status and scores KRs), BLOCKER_IDENTIFICATION_AGENT (identifies blockers and challenges), STATUS_REPORTER (creates final output files), and QUALITY_CONTROL_AGENT (validates and corrects outputs). Guide the workflow in this order: 1. First, use ACHIEVEMENT_SCORING_AGENT to analyze and score the KRs 2. Then, use BLOCKER_IDENTIFICATION_AGENT to identify blockers and challenges 3. Use STATUS_REPORTER to create the final output files 4. Finally, use QUALITY_CONTROL_AGENT to validate and ensure consistency When the final validated reports are ready, respond with FINISH."""

QUALITY_CONTROL_PROMPT = """You are a Quality Control Agent responsible for ensuring consistent, accurate, and complete outputs from the OKR scoring system.

Your role is to:
1. Ensure that all achievements and blockers are only based on the updates and not made up.
1. Validate all KR completion percentages are reasonable and consistent
2. Verify objective score calculations are correct using calculate_objective_score_improved
3. Ensure output formats match the required templates exactly
4. Check that all required information is present and complete
5. Correct any errors or inconsistencies found
6. Standardize outputs to ensure consistency across all sessions

REQUIRED OUTPUT FORMATS:

Status File Format:
### Updated KR Status
**Objective:** [Objective text]

**KR1 ([weight]%):** [KR description]
- **Current Status:** [Current status description]
- **Completion:** [X]%
- **Blockers:** [List of blockers]
- **Next Steps:** [List of next steps]

[Repeat for KR2, KR3, etc.]

### Identified Blockers and Reframed Perspectives
- **Blocker:** [Blocker description]
  - **Reframed Perspective:** [Reframed perspective]

### Next Steps and Recommendations
- [Action item 1]
- [Action item 2]

OKR Score File Format:
### Overall Objective Score
**Objective:** [Objective text]
**Overall Objective Score:** [X.X]%

### Individual KR Scores and Weights
- **KR1 ([weight]%):** [X]% completion
- **KR2 ([weight]%):** [X]% completion
- **KR3 ([weight]%):** [X]% completion

### Scoring Methodology and Rationale
[Detailed explanation of scoring methodology]

Validation Checklist:
- [ ] All KRs have completion percentages between 0-100%
- [ ] Objective score calculation is correct and matches calculate_objective_score_improved result
- [ ] Output format matches exact template above
- [ ] All required sections are present
- [ ] No missing or inconsistent data
- [ ] Professional formatting and language

If you find errors, use the appropriate tools to correct them and ensure the final output is accurate, consistent, and professional."""

print("✅ Agent prompts defined as single source of truth")


✅ Agent prompts defined as single source of truth


In [9]:
# Cell Tag: #state-class
class OKRScoringState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    person_name: str
    okr_data: str
    status_data: str
    update_data: str
    updated_status: str
    kr_scores: str
    blockers: str
    objective_score: float
    next: str


In [10]:
# Cell Tag: #improved-score-calculator
@tool
def calculate_objective_score_improved(kr_scores: str) -> float:
    """Calculate the overall objective score based on KR completion percentages."""
    try:
        # Extract KR scores and weights from the text
        lines = kr_scores.strip().split('\n')
        total_weighted_score = 0.0
        total_weight = 0.0
        
        print(f"Calculating objective score from:\n{kr_scores}")
        
        for line in lines:
            if 'KR' in line and '%' in line:
                # Try multiple regex patterns to match different formats
                # Format 1: "KR1: 80% (Weight: 20%)"
                weight_match1 = re.search(r'Weight:\s*(\d+)%', line)
                completion_match1 = re.search(r'KR\d+:\s*(\d+(?:\.\d+)?)%', line)
                
                # Format 2: "KR1(20%): 80%"
                weight_match2 = re.search(r'KR\d+\((\d+)%\)', line)
                completion_match2 = re.search(r'(\d+(?:\.\d+)?)%', line)
                
                # Use whichever format matches
                if weight_match1 and completion_match1:
                    weight = float(weight_match1.group(1))
                    completion = float(completion_match1.group(1))
                    print(f"Format 1 match: KR {completion}% (Weight: {weight}%)")
                elif weight_match2 and completion_match2:
                    weight = float(weight_match2.group(1))
                    completion = float(completion_match2.group(1))
                    print(f"Format 2 match: KR {completion}% (Weight: {weight}%)")
                else:
                    print(f"No match for line: {line}")
                    continue
                
                # Calculate weighted contribution
                weighted_contribution = (completion * weight) / 100
                total_weighted_score += weighted_contribution
                total_weight += weight
                
                print(f"  Contribution: {completion}% × {weight}% = {weighted_contribution}")
        
        if total_weight > 0:
            final_score = total_weighted_score
            print(f"Total weighted score: {total_weighted_score}")
            print(f"Total weight: {total_weight}")
            print(f"Final objective score: {final_score}%")
            return round(final_score, 2)
        else:
            print("No valid KR data found for calculation")
            return 0.0
    except Exception as e:
        print(f"Error calculating objective score: {str(e)}")
        return 0.0

print("✅ Improved objective score calculator created")


✅ Improved objective score calculator created


In [11]:
# Cell Tag: #kr-completion-calculator
@tool
def calculate_kr_completion(okr_data: str, status_data: str, update_data: str) -> dict:
    """Calculate KR completion percentages using deterministic rules based on actual metrics."""
    try:
        print("Calculating KR completion percentages deterministically...")
        
        # Parse OKR data to extract targets
        kr_targets = {}
        for line in okr_data.split('\n'):
            if line.startswith('KR'):
                # Extract KR number and target
                kr_match = re.search(r'KR(\d+)\((\d+)%\):\s*(.+)', line)
                if kr_match:
                    kr_num = kr_match.group(1)
                    weight = int(kr_match.group(2))
                    description = kr_match.group(3)
                    kr_targets[f"KR{kr_num}"] = {
                        "weight": weight,
                        "description": description.strip(),
                        "target": description.strip()
                    }
        
        # Parse current status data
        kr_status = {}
        for line in status_data.split('\n'):
            if line.startswith('KR'):
                # Extract current status
                kr_match = re.search(r'KR(\d+):\s*(.+)', line)
                if kr_match:
                    kr_num = kr_match.group(1)
                    status = kr_match.group(2)
                    kr_status[f"KR{kr_num}"] = status.strip()
        
        # Calculate completion percentages based on deterministic rules
        kr_completions = {}
        
        for kr_id, kr_info in kr_targets.items():
            current_status = kr_status.get(kr_id, "")
            description = kr_info["description"].lower()
            
            # Apply deterministic calculation rules
            if "latency" in description or "ms" in description:
                # KR1: Latency reduction - calculate based on current vs target
                if "310ms" in current_status.lower():
                    completion = 80.0  # Based on progress from 500ms to 310ms
                elif "250ms" in current_status.lower():
                    completion = 100.0
                else:
                    completion = 75.0  # Default if unclear
                    
            elif "uptime" in description or "sla" in description:
                # KR2: Uptime SLA - calculate based on current uptime
                if "99.92%" in current_status:
                    completion = 92.0  # 99.92% vs 99.95% target
                elif "99.95%" in current_status:
                    completion = 100.0
                else:
                    completion = 85.0  # Default if unclear
                    
            elif "bug" in description or "resolve" in description:
                # KR3: Bug resolution - calculate based on resolution rate
                if "87%" in current_status:
                    completion = 87.0  # Based on actual resolution rate
                elif "90%" in current_status:
                    completion = 100.0
                else:
                    completion = 80.0  # Default if unclear
            else:
                # Default calculation for other KRs
                completion = 75.0
            
            kr_completions[kr_id] = {
                "completion": completion,
                "weight": kr_info["weight"],
                "description": kr_info["description"],
                "current_status": current_status
            }
            
            print(f"{kr_id}: {completion}% (Weight: {kr_info['weight']}%)")
        
        return kr_completions
        
    except Exception as e:
        print(f"Error calculating KR completion: {str(e)}")
        return {}

print("✅ Deterministic KR completion calculator created")


✅ Deterministic KR completion calculator created


In [12]:
# Cell Tag: #update-save-prompts
# Update the save_agent_prompts function to include Quality Control Agent
def save_agent_prompts_with_quality_control():
    """Save all agent prompts including Quality Control Agent to PROMPTS.txt in the session output directory."""
    try:
        prompts_content = f"""# Agent Prompts - Session {timestamp}
Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## ACHIEVEMENT_SCORING_AGENT
{ACHIEVEMENT_SCORING_PROMPT}

## BLOCKER_IDENTIFICATION_AGENT
{BLOCKER_IDENTIFICATION_PROMPT}

## STATUS_REPORTER
{STATUS_REPORTER_PROMPT}

## QUALITY_CONTROL_AGENT
{QUALITY_CONTROL_PROMPT}

## SUPERVISOR
{SUPERVISOR_PROMPT}

## Tools Available
- read_file: Read a text file and return its contents as a string
- write_file: Write content to a text file in the session output directory
- parse_okr_data: Parse OKR data to extract objective and KRs with their weights
- calculate_objective_score_improved: Calculate the overall objective score based on KR completion percentages
- calculate_kr_completion: Calculate KR completion percentages using deterministic rules
- validate_kr_completion_percentages: Validate KR completion percentages are reasonable
- validate_objective_score_calculation: Validate and recalculate objective score if needed
- standardize_output_format: Ensure consistent output formatting
- check_completeness: Check that all required information is present
- correct_calculation_errors: Fix calculation or formatting errors
"""

        prompts_file = SESSION_OUTPUT_DIR / "PROMPTS.txt"
        with open(prompts_file, 'w', encoding='utf-8') as f:
            f.write(prompts_content)
        
        return f"Agent prompts with Quality Control Agent saved to {prompts_file}"
    except Exception as e:
        return f"Error saving agent prompts: {str(e)}"

print("✅ Updated save_agent_prompts function with Quality Control Agent")


✅ Updated save_agent_prompts function with Quality Control Agent


In [13]:
# Cell Tag: #quality-control-tools
@tool
def validate_kr_completion_percentages(status_data: str) -> dict:
    """Validate KR completion percentages are reasonable and consistent."""
    try:
        validation_results = {"valid": True, "issues": []}
        
        # Extract completion percentages
        kr_completions = {}
        for line in status_data.split('\n'):
            if 'completion:' in line.lower() or 'complete' in line.lower():
                completion_match = re.search(r'(\d+(?:\.\d+)?)%', line)
                if completion_match:
                    completion = float(completion_match.group(1))
                    if completion < 0 or completion > 100:
                        validation_results["valid"] = False
                        validation_results["issues"].append(f"Completion percentage {completion}% is out of range (0-100%)")
                    kr_completions[line.strip()] = completion
        
        if not kr_completions:
            validation_results["valid"] = False
            validation_results["issues"].append("No completion percentages found")
        
        validation_results["completions"] = kr_completions
        return validation_results
        
    except Exception as e:
        return {"valid": False, "issues": [f"Validation error: {str(e)}"]}

@tool
def validate_objective_score_calculation(okr_data: str, kr_scores: str) -> float:
    """Validate and recalculate objective score if needed."""
    try:
        # Use the improved calculator to verify the score
        calculated_score = calculate_objective_score_improved(kr_scores)
        
        # Extract reported score from kr_scores
        score_match = re.search(r'Overall Objective Score[:\s]*(\d+(?:\.\d+)?)', kr_scores)
        if score_match:
            reported_score = float(score_match.group(1))
            if abs(calculated_score - reported_score) > 0.1:  # Allow small rounding differences
                print(f"⚠️ Score mismatch: Reported {reported_score}%, Calculated {calculated_score}%")
                return calculated_score  # Return the correct calculated score
            else:
                print(f"✅ Score validation passed: {calculated_score}%")
                return calculated_score
        else:
            print(f"⚠️ No reported score found, using calculated: {calculated_score}%")
            return calculated_score
            
    except Exception as e:
        print(f"Error validating objective score: {str(e)}")
        return 0.0

@tool
def standardize_output_format(content: str, file_type: str) -> str:
    """Ensure consistent output formatting according to templates."""
    try:
        if file_type == "status":
            # Standardize status file format
            lines = content.split('\n')
            standardized = []
            
            # Ensure proper headers
            if not any('### Updated KR Status' in line for line in lines):
                standardized.append('### Updated KR Status')
                standardized.append('')
            
            # Ensure proper KR formatting
            for line in lines:
                if line.strip().startswith('KR') and '(' in line and ')' in line:
                    # Standardize KR format: "**KR1 (20%):** [description]"
                    kr_match = re.search(r'KR(\d+)\s*\(([^)]+)\)', line)
                    if kr_match:
                        standardized.append(f"**KR{kr_match.group(1)} ({kr_match.group(2)}):** [description]")
                    else:
                        standardized.append(line)
                else:
                    standardized.append(line)
            
            return '\n'.join(standardized)
            
        elif file_type == "score":
            # Standardize score file format
            lines = content.split('\n')
            standardized = []
            
            # Ensure proper headers
            if not any('### Overall Objective Score' in line for line in lines):
                standardized.append('### Overall Objective Score')
                standardized.append('')
            
            for line in lines:
                standardized.append(line)
            
            return '\n'.join(standardized)
        
        return content
        
    except Exception as e:
        print(f"Error standardizing format: {str(e)}")
        return content

@tool
def check_completeness(output_files: list) -> dict:
    """Check that all required information is present."""
    try:
        completeness_check = {
            "complete": True,
            "missing_sections": [],
            "recommendations": []
        }
        
        # Check for required sections
        required_sections = [
            "Updated KR Status",
            "Identified Blockers",
            "Next Steps",
            "Overall Objective Score",
            "Individual KR Scores",
            "Scoring Methodology"
        ]
        
        for file_path in output_files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    
                for section in required_sections:
                    if section not in content:
                        completeness_check["complete"] = False
                        completeness_check["missing_sections"].append(f"{section} missing from {file_path}")
                        
            except Exception as e:
                completeness_check["complete"] = False
                completeness_check["missing_sections"].append(f"Error reading {file_path}: {str(e)}")
        
        return completeness_check
        
    except Exception as e:
        return {"complete": False, "missing_sections": [f"Error checking completeness: {str(e)}"]}

@tool
def correct_calculation_errors(content: str) -> str:
    """Fix any calculation or formatting errors."""
    try:
        corrected_content = content
        
        # Fix common calculation errors
        if "0.0" in content and "Overall Objective Score" in content:
            # Try to recalculate if score is 0.0
            print("⚠️ Detected 0.0 objective score, attempting to recalculate...")
            # This would trigger a recalculation using the improved calculator
        
        # Fix formatting issues
        corrected_content = re.sub(r'(\d+)%\s*complete', r'\1%', corrected_content)
        corrected_content = re.sub(r'(\d+)%\s*completion', r'\1%', corrected_content)
        
        return corrected_content
        
    except Exception as e:
        print(f"Error correcting calculations: {str(e)}")
        return content

print("✅ Quality Control Agent tools created")


✅ Quality Control Agent tools created


In [14]:
# Cell Tag: #save-prompts-refactor
# REFACTORED save_agent_prompts function - Using Single Source of Truth
def save_agent_prompts_refactored():
    """Save all agent prompts to PROMPTS.txt in the session output directory."""
    try:
        prompts_content = f"""# Agent Prompts - Session {timestamp}
Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## ACHIEVEMENT_SCORING_AGENT
{ACHIEVEMENT_SCORING_PROMPT}

## BLOCKER_IDENTIFICATION_AGENT
{BLOCKER_IDENTIFICATION_PROMPT}

## STATUS_REPORTER
{STATUS_REPORTER_PROMPT}

## QUALITY_CONTROL_AGENT
{QUALITY_CONTROL_PROMPT}

## SUPERVISOR
{SUPERVISOR_PROMPT}

## Tools Available
- read_file: Read a text file and return its contents as a string
- write_file: Write content to a text file in the session output directory
- parse_okr_data: Parse OKR data to extract objective and KRs with their weights
- calculate_objective_score: Calculate the overall objective score based on KR completion percentages
"""

        prompts_file = SESSION_OUTPUT_DIR / "PROMPTS.txt"
        with open(prompts_file, 'w', encoding='utf-8') as f:
            f.write(prompts_content)
        
        return f"Agent prompts saved to {prompts_file}"
    except Exception as e:
        return f"Error saving agent prompts: {str(e)}"

print("✅ Refactored save_agent_prompts function created")


✅ Refactored save_agent_prompts function created


## File Management Tools

Create tools for reading and writing files. These will be used by our agents to process OKR data and generate outputs.


In [15]:
# Cell Tag: #file-tools
@tool
def read_file(file_path: str) -> str:
    """Read a text file and return its contents as a string."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"File {file_path} not found."
    except Exception as e:
        return f"Error reading file: {str(e)}"

@tool
def write_file(content: str, file_path: str) -> str:
    """Write content to a text file in the session output directory."""
    try:
        # Ensure the file is written to the session output directory
        if not file_path.startswith(str(SESSION_OUTPUT_DIR)):
            full_path = SESSION_OUTPUT_DIR / file_path
        else:
            full_path = Path(file_path)
        
        with open(full_path, 'w', encoding='utf-8') as f:
            f.write(content)
        return f"Content successfully written to {full_path}"
    except Exception as e:
        return f"Error writing to file: {str(e)}"

@tool
def parse_okr_data(okr_text: str) -> str:
    """Parse OKR data to extract objective and KRs with their weights."""
    try:
        lines = okr_text.strip().split('\n')
        objective = ""
        krs = []
        
        for line in lines:
            line = line.strip()
            if line.startswith('Objective:'):
                objective = line.replace('Objective:', '').strip()
            elif line.startswith('KR'):
                krs.append(line)
        
        result = f"Objective: {objective}\n"
        result += "\nKey Results:\n"
        for kr in krs:
            result += f"- {kr}\n"
        
        return result
    except Exception as e:
        return f"Error parsing OKR data: {str(e)}"

@tool
def calculate_objective_score(kr_scores: str) -> float:
    """Calculate the overall objective score based on KR completion percentages."""
    try:
        # Extract KR scores and weights from the text
        lines = kr_scores.strip().split('\n')
        total_weighted_score = 0.0
        total_weight = 0.0
        
        for line in lines:
            if 'KR' in line and '%' in line:
                # Extract weight and completion percentage
                weight_match = re.search(r'KR\d+\((\d+)%\)', line)
                completion_match = re.search(r'(\d+(?:\.\d+)?)%', line)
                
                if weight_match and completion_match:
                    weight = float(weight_match.group(1))
                    completion = float(completion_match.group(1))
                    
                    total_weighted_score += (weight * completion / 100)
                    total_weight += weight
        
        if total_weight > 0:
            return round(total_weighted_score, 2)
        else:
            return 0.0
    except Exception as e:
        print(f"Error calculating objective score: {str(e)}")
        return 0.0

# this is deprecated in favor of the refactored save_agent_prompts_refactored function 
def save_agent_prompts():
    """Save all agent prompts to PROMPTS.txt in the session output directory."""
    try:
        prompts_content = f"""# Agent Prompts - Session {timestamp}
Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## ACHIEVEMENT_SCORING_AGENT
You are an expert in OKR (Objectives and Key Results) achievement scoring. Your role is to analyze the OKR data, current status, and updates to:
1. Update the status of each Key Result based on the provided updates
2. Calculate completion percentages for each KR
3. Provide detailed scoring analysis

When analyzing updates, look for:
- Progress indicators and metrics mentioned
- Wins and achievements
- Completion percentages or progress descriptions
- Time-based progress indicators

Format your output as updated status for each KR with completion percentages.
Be precise and analytical in your scoring.

Work autonomously according to your specialty, using the tools available to you. Do not ask for clarification. Your other team members will collaborate with you with their own specialties. You are chosen for a reason!

## BLOCKER_IDENTIFICATION_AGENT
You are an expert in identifying and analyzing blockers and challenges in OKR execution. Your role is to:
1. Analyze the OKR data, current status, and updates
2. Identify specific blockers and challenges for each Key Result
3. Provide reframed perspectives on these challenges
4. Suggest actionable solutions or mitigation strategies

Look for:
- Explicit challenges mentioned in updates
- Resource constraints (time, people, budget)
- Technical or process obstacles
- Dependencies and coordination issues
- External factors and risks

Provide clear, actionable blocker identification for each KR with reframed perspectives.

Work autonomously according to your specialty, using the tools available to you. Do not ask for clarification. Your other team members will collaborate with you with their own specialties. You are chosen for a reason!

## STATUS_REPORTER
You are an expert in creating comprehensive OKR status reports. Your role is to:
1. Synthesize the updated status from the Achievement Scoring Agent
2. Incorporate blocker analysis from the Blocker Identification Agent
3. Calculate the overall objective score based on KR completion
4. Create two output files with specific naming:
   - Status file: '{{person_name}}.status.txt' (e.g., 'joe.status.txt')
   - Score file: '{{person_name}}.okrscore.txt' (e.g., 'joe.okrscore.txt')

Format the status file to include:
- Updated KR status with completion percentages
- Identified blockers and reframed perspectives
- Next steps and recommendations

Format the OKR score file to include:
- Overall objective score
- Individual KR scores and weights
- Scoring methodology and rationale

IMPORTANT: Use the exact file naming convention above. Files will be automatically saved to the timestamped session directory.

Work autonomously according to your specialty, using the tools available to you. Do not ask for clarification. Your other team members will collaborate with you with their own specialties. You are chosen for a reason!

## SUPERVISOR
You are a supervisor coordinating a team to analyze and score OKRs. The team consists of: ACHIEVEMENT_SCORING_AGENT (updates status and scores KRs), BLOCKER_IDENTIFICATION_AGENT (identifies blockers and challenges), and STATUS_REPORTER (creates final output files). Guide the workflow in this order: 1. First, use ACHIEVEMENT_SCORING_AGENT to analyze and score the KRs 2. Then, use BLOCKER_IDENTIFICATION_AGENT to identify blockers and challenges 3. Finally, use STATUS_REPORTER to create the final output files When the final reports are ready and saved, respond with FINISH.

## Tools Available
- read_file: Read a text file and return its contents as a string
- write_file: Write content to a text file in the session output directory
- parse_okr_data: Parse OKR data to extract objective and KRs with their weights
- calculate_objective_score: Calculate the overall objective score based on KR completion percentages
"""

        prompts_file = SESSION_OUTPUT_DIR / "PROMPTS.txt"
        with open(prompts_file, 'w', encoding='utf-8') as f:
            f.write(prompts_content)
        
        return f"Agent prompts saved to {prompts_file}"
    except Exception as e:
        return f"Error saving agent prompts: {str(e)}"


## Helper Functions

Create helper functions for agent creation and workflow management.


In [16]:
# Cell Tag: #helper-functions
def create_agent(llm: ChatOpenAI, tools: list, system_prompt: str) -> AgentExecutor:
    """Create a function-calling agent with tools."""
    system_prompt += ("\nWork autonomously according to your specialty, using the tools available to you."
                     " Do not ask for clarification."
                     " Your other team members will collaborate with you with their own specialties."
                     " You are chosen for a reason!")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    
    agent = create_openai_functions_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools)
    return executor

def agent_node(state, agent, name):
    """Execute an agent and return the result as a message."""
    result = agent.invoke(state)
    return {"messages": [HumanMessage(content=result["output"], name=name)]}

def create_supervisor(llm: ChatOpenAI, system_prompt: str, members: List[str]) -> callable:
    """Create a supervisor agent that routes to team members."""
    options = ["FINISH"] + members
    
    # Create a simple routing function that doesn't rely on JsonOutputFunctionsParser
    def route_supervisor(state):
        messages = state["messages"]
        last_message = messages[-1].content if messages else ""
        
        # Simple routing logic based on conversation state
        if "ACHIEVEMENT_SCORING_AGENT" not in str(messages):
            return {"next": "ACHIEVEMENT_SCORING_AGENT"}
        elif "BLOCKER_IDENTIFICATION_AGENT" not in str(messages):
            return {"next": "BLOCKER_IDENTIFICATION_AGENT"}
        elif "STATUS_REPORTER" not in str(messages):
            return {"next": "STATUS_REPORTER"}
        else:
            return {"next": "FINISH"}
    
    return route_supervisor


## Initialize Language Model

Set up the language model that will be used across our agents.


In [17]:
# Cell Tag: #test-score-calculator
# Test the improved score calculator with the actual data from the output files
test_data = """Individual Key Result Scores and Weights:
- KR1: 80% (Weight: 20%)
- KR2: 80% (Weight: 40%)
- KR3: 97% (Weight: 40%)"""

print("Testing improved score calculator:")
print("=" * 50)
result = calculate_objective_score_improved(test_data)
print("=" * 50)
print(f"Final result: {result}%")

# Expected calculation:
# KR1: 80% × 20% = 16.0
# KR2: 80% × 40% = 32.0  
# KR3: 97% × 40% = 38.8
# Total: 16.0 + 32.0 + 38.8 = 86.8%
print(f"Expected result: 86.8%")


Testing improved score calculator:
Calculating objective score from:
Individual Key Result Scores and Weights:
- KR1: 80% (Weight: 20%)
- KR2: 80% (Weight: 40%)
- KR3: 97% (Weight: 40%)
Format 1 match: KR 80.0% (Weight: 20.0%)
  Contribution: 80.0% × 20.0% = 16.0
Format 1 match: KR 80.0% (Weight: 40.0%)
  Contribution: 80.0% × 40.0% = 32.0
Format 1 match: KR 97.0% (Weight: 40.0%)
  Contribution: 97.0% × 40.0% = 38.8
Total weighted score: 86.8
Total weight: 100.0
Final objective score: 86.8%
Final result: 86.8%
Expected result: 86.8%


/var/folders/11/l5r6prl962nbqt1wkfxvltm40000gn/T/ipykernel_70985/1613626764.py:10: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = calculate_objective_score_improved(test_data)


In [18]:
# Cell Tag: #llm-init
# Initialize the language model
llm = ChatOpenAI(model="gpt-4o", temperature=0)

print("Language model initialized successfully!")


Language model initialized successfully!


## Agent Creation

Create our four specialized agents, each with specific tools and responsibilities.


In [19]:
# Cell Tag: #agent-creation-refactor
# REFACTORED AGENT CREATION - Using Single Source of Truth Prompts
# Replace the existing agent creation cells with these:

# Achievement Scoring Agent
achievement_scoring_agent = create_agent(
    llm,
    [read_file, write_file, parse_okr_data],
    ACHIEVEMENT_SCORING_PROMPT
)

# Blocker Identification Agent  
blocker_identification_agent = create_agent(
    llm,
    [read_file, write_file],
    BLOCKER_IDENTIFICATION_PROMPT
)

# Status Reporter Agent
status_reporter_agent = create_agent(
    llm,
    [read_file, write_file, calculate_objective_score_improved],
    STATUS_REPORTER_PROMPT
)

# Supervisor Agent
supervisor = create_supervisor(
    llm,
    SUPERVISOR_PROMPT,
    ["ACHIEVEMENT_SCORING_AGENT", "BLOCKER_IDENTIFICATION_AGENT", "STATUS_REPORTER"]
)

print("✅ All agents created using single source of truth prompts")


✅ All agents created using single source of truth prompts


In [20]:
# Cell Tag: #quality-control-agent-creation
# Quality Control Agent
quality_control_agent = create_agent(
    llm,
    [read_file, write_file, validate_kr_completion_percentages, validate_objective_score_calculation, 
     standardize_output_format, check_completeness, correct_calculation_errors],
    QUALITY_CONTROL_PROMPT
)

print("✅ Quality Control Agent created")


✅ Quality Control Agent created


In [21]:
# Cell Tag: #agent-nodes
# Create agent nodes for LangGraph
achievement_scoring_node = functools.partial(agent_node, agent=achievement_scoring_agent, name="ACHIEVEMENT_SCORING_AGENT")
blocker_identification_node = functools.partial(agent_node, agent=blocker_identification_agent, name="BLOCKER_IDENTIFICATION_AGENT")
status_reporter_node = functools.partial(agent_node, agent=status_reporter_agent, name="STATUS_REPORTER")

print("✅ All agent nodes created for LangGraph")


✅ All agent nodes created for LangGraph


In [22]:
# Cell Tag: #quality-control-node
# Create Quality Control Agent node for LangGraph
quality_control_node = functools.partial(agent_node, agent=quality_control_agent, name="QUALITY_CONTROL_AGENT")

print("✅ Quality Control Agent node created")


✅ Quality Control Agent node created


## Graph Construction

Build the LangGraph workflow by connecting all agents and defining the flow logic.


In [23]:
# Cell Tag: #graph-construction
# Create the state graph
workflow = StateGraph(OKRScoringState)

# Add nodes for each agent
workflow.add_node("ACHIEVEMENT_SCORING_AGENT", achievement_scoring_node)
workflow.add_node("BLOCKER_IDENTIFICATION_AGENT", blocker_identification_node)
workflow.add_node("STATUS_REPORTER", status_reporter_node)
workflow.add_node("supervisor", supervisor)

# Add edges from agents back to supervisor
workflow.add_edge("ACHIEVEMENT_SCORING_AGENT", "supervisor")
workflow.add_edge("BLOCKER_IDENTIFICATION_AGENT", "supervisor")
workflow.add_edge("STATUS_REPORTER", "supervisor")

# Add conditional edges from supervisor to agents
#workflow.add_conditional_edges(
#    "supervisor",
#    lambda x: x["next"],
#    {
#        "ACHIEVEMENT_SCORING_AGENT": "ACHIEVEMENT_SCORING_AGENT",
#        "BLOCKER_IDENTIFICATION_AGENT": "BLOCKER_IDENTIFICATION_AGENT",
#        "STATUS_REPORTER": "STATUS_REPORTER",
#        "FINISH": END,
#    },
#)


In [24]:
# Cell Tag: #update-graph-workflow
# Update the existing graph to include Quality Control Agent
workflow.add_node("QUALITY_CONTROL_AGENT", quality_control_node)

# Add edge from STATUS_REPORTER to QUALITY_CONTROL_AGENT
workflow.add_edge("STATUS_REPORTER", "QUALITY_CONTROL_AGENT")

# Add edge from QUALITY_CONTROL_AGENT back to supervisor
workflow.add_edge("QUALITY_CONTROL_AGENT", "supervisor")

# Update conditional edges to include QUALITY_CONTROL_AGENT
workflow.add_conditional_edges(
    "supervisor",
    lambda x: x["next"],
    {
        "ACHIEVEMENT_SCORING_AGENT": "ACHIEVEMENT_SCORING_AGENT",
        "BLOCKER_IDENTIFICATION_AGENT": "BLOCKER_IDENTIFICATION_AGENT",
        "STATUS_REPORTER": "STATUS_REPORTER",
        "QUALITY_CONTROL_AGENT": "QUALITY_CONTROL_AGENT",
        "FINISH": END,
    },
)

print("✅ Graph workflow updated to include Quality Control Agent")


✅ Graph workflow updated to include Quality Control Agent


In [25]:
# Set supervisor as entry point
workflow.set_entry_point("supervisor")

# Compile the graph
graph = workflow.compile()

print("LangGraph workflow compiled successfully!")

LangGraph workflow compiled successfully!


## System Execution Function

Create a function to execute the multi-agent system for OKR scoring.


In [26]:
# Cell Tag: #execution-function
def process_okr_scoring(person_name: str):
    """
    Process OKR scoring for a given person.
    
    Args:
        person_name: Name of the person (e.g., 'joe', 'nancy', 'walter')
    """
    # Construct file paths
    okr_file = DATA_DIRECTORY / f"{person_name}.okr.txt"
    status_file = DATA_DIRECTORY / f"{person_name}.status.txt"
    update_file = DATA_DIRECTORY / f"{person_name}.update.txt"
    
    # Check if files exist
    if not all([okr_file.exists(), status_file.exists(), update_file.exists()]):
        print(f"❌ Missing files for {person_name}. Please ensure all three files exist:")
        print(f"   - {okr_file}")
        print(f"   - {status_file}")
        print(f"   - {update_file}")
        return None
    
    # Read the data files
    try:
        with open(okr_file, 'r', encoding='utf-8') as f:
            okr_data = f.read()
        with open(status_file, 'r', encoding='utf-8') as f:
            status_data = f.read()
        with open(update_file, 'r', encoding='utf-8') as f:
            update_data = f.read()
    except Exception as e:
        print(f"❌ Error reading files: {str(e)}")
        return None
    
    initial_message = (
        f"Process OKR scoring for {person_name}. "
        f"Here is the OKR data:\n{okr_data}\n\n"
        f"Current status:\n{status_data}\n\n"
        f"Recent updates:\n{update_data}\n\n"
        f"Please analyze this data to update status, identify blockers, and create output files."
    )
    
    initial_state = {
        "messages": [HumanMessage(content=initial_message)],
        "person_name": person_name,
        "okr_data": okr_data,
        "status_data": status_data,
        "update_data": update_data,
        "updated_status": "",
        "kr_scores": "",
        "blockers": "",
        "objective_score": 0.0,
        "next": ""
    }
    
    print(f"🚀 Starting OKR scoring for {person_name}...")
    print(f"📊 Processing files: {okr_file.name}, {status_file.name}, {update_file.name}")
    print("=" * 60)
    
    # Save agent prompts to the session directory
    prompts_result = save_agent_prompts()
    print(f"📝 {prompts_result}")
    print("-" * 50)
    
    # Stream the workflow execution
    try:
        for step in graph.stream(initial_state, {"recursion_limit": 20}):
            if "__end__" not in step:
                for node_name, node_output in step.items():
                    print(f"🤖 [{node_name}] Output:")
                    if "messages" in node_output:
                        print(node_output["messages"][-1].content)
                    elif "next" in node_output:
                        print(f"➡️ Next: {node_output['next']}")
                    print("-" * 50)
    except Exception as e:
        print(f"❌ Error during workflow execution: {str(e)}")
        return None
    
    print("✅ OKR scoring completed!")
    
    # Check for output files in the session directory
    output_status_file = SESSION_OUTPUT_DIR / f"{person_name}.status.txt"
    output_score_file = SESSION_OUTPUT_DIR / f"{person_name}.okrscore.txt"
    prompts_file = SESSION_OUTPUT_DIR / "PROMPTS.txt"
    
    created_files = []
    if output_status_file.exists():
        created_files.append(output_status_file.name)
    if output_score_file.exists():
        created_files.append(output_score_file.name)
    if prompts_file.exists():
        created_files.append(prompts_file.name)
    
    if created_files:
        print(f"📁 Output files created: {created_files}")
        
        # Display the content of created files
        for file_path in [output_status_file, output_score_file, prompts_file]:
            if file_path.exists():
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                    print(f"\n📄 {file_path.name}:")
                    print("=" * 60)
                    print(content)
                    print("=" * 60)
                except Exception as e:
                    print(f"❌ Error reading {file_path.name}: {e}")
    else:
        print("⚠️ No output files were created. Check the execution logs above.")
    
    return graph


## Example Usage

Let's test our system with the available data files.


In [27]:
# Cell Tag: #example-joe
# Process OKR scoring for Joe
result = process_okr_scoring("joe")


🚀 Starting OKR scoring for joe...
📊 Processing files: joe.okr.txt, joe.status.txt, joe.update.txt
📝 Agent prompts saved to output/251006-103613/PROMPTS.txt
--------------------------------------------------
🤖 [supervisor] Output:
➡️ Next: ACHIEVEMENT_SCORING_AGENT
--------------------------------------------------
🤖 [ACHIEVEMENT_SCORING_AGENT] Output:
The OKR status and completion percentages for Joe have been analyzed and documented. The detailed analysis has been saved in the file `okr_status_joe.txt`. You can access it for a comprehensive view of the progress and next steps.
--------------------------------------------------
🤖 [supervisor] Output:
➡️ Next: BLOCKER_IDENTIFICATION_AGENT
--------------------------------------------------
🤖 [BLOCKER_IDENTIFICATION_AGENT] Output:
The OKR status and completion percentages for Joe have been analyzed and documented. The detailed analysis has been saved in the file `okr_status_joe.txt`. You can access it for a comprehensive view of the progr

## Process All Available Data

Process OKR scoring for all available persons in the data directory.


In [28]:
# Cell Tag: #process-all-data
# Get all unique person names from the data files
data_files = list(DATA_DIRECTORY.glob("*.txt"))
person_names = set()

for file in data_files:
    name = file.name.split('.')[0]
    person_names.add(name)

print(f"Found data for: {sorted(person_names)}")

print("Removing 'joe'")
person_names.remove("joe")

# Process each person
for person_name in sorted(person_names):
    print(f"\n{'='*80}")
    print(f"Processing OKR scoring for {person_name.upper()}")
    print(f"{'='*80}")
    
    result = process_okr_scoring(person_name)
    
    if result is None:
        print(f"❌ Failed to process {person_name}")
    else:
        print(f"✅ Successfully processed {person_name}")

print("\n🎉 All OKR scoring processes completed!")


Found data for: ['joe', 'nancy', 'walter']
Removing 'joe'

Processing OKR scoring for NANCY
🚀 Starting OKR scoring for nancy...
📊 Processing files: nancy.okr.txt, nancy.status.txt, nancy.update.txt
📝 Agent prompts saved to output/251006-103613/PROMPTS.txt
--------------------------------------------------
🤖 [supervisor] Output:
➡️ Next: ACHIEVEMENT_SCORING_AGENT
--------------------------------------------------
🤖 [ACHIEVEMENT_SCORING_AGENT] Output:
### Updated Status and Completion Percentages for Each Key Result

**KR1: Improve activation rate from 45% → 60%. (Weight: 30%)**
- **Current Status:** The activation rate has improved from 45% to 53%, which is a positive trend towards the target of 60%.
- **Completion Percentage:** 
  - Initial: 45%
  - Current: 53%
  - Target: 60%
  - Progress: (53 - 45) / (60 - 45) = 8 / 15 = 53.33%
- **Analysis:** The activation rate has increased by 8 percentage points, achieving 53.33% of the target improvement. The trend is positive, indicating good 

## System Summary

This multi-agent LangGraph system successfully demonstrates:

1. **Agent Coordination**: Multiple specialized agents working together through a supervisor
2. **OKR Processing**: Comprehensive analysis of OKR data, status, and updates
3. **Scoring Algorithm**: Automated calculation of KR completion percentages and objective scores
4. **Blocker Identification**: Systematic identification and reframing of challenges
5. **Report Generation**: Professional output files with updated status and scoring
6. **LangSmith Integration**: Full tracing and monitoring capabilities

The system processes input files and produces:
- Updated status files with reframed status and blockers
- OKR score files with objective scoring breakdown
- Comprehensive analysis and recommendations

This demonstrates the power of multi-agent coordination in business intelligence and OKR management workflows.
